## DIPm-TV simulation pipeline

| Scenario | Structure | Channels | Tilt range | Reference |
|---|---|---|---|---|
| **EDX** | Cube + sphere hole + ellipsoid hole | 3 (cube / ellipsoid / sphere) | −40 : 5 : +40  (17 proj.) | arXiv:2606.10547 |
| **EELS** | Hollow core-shell nanocube with pore | 2 (shell / core) | −70 : 17.5 : +70  (9 proj.) | arXiv:2606.10893 |

In [ ]:
import sys
sys.path.insert(0, '../../Src')

from radon import Radon3D
from dipm_tv import CNN3D, run_dipm_tv, preprocess_sinograms
from utils import (
    norm_slice, vol_slice, embed_sino,
    nrmse, ssim_score,
    rgb_channel, rgb_two_ch,
    panel_results, show_channels,
    paint_sphere, paint_ellipsoid, rotate_rebinarize,
    norm_percentile,
)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
plt.rcParams['image.cmap'] = 'gray'
print(f'Device: {DEVICE}')

INPUT_DEPTH = 32
THETA_REF   = np.arange(0, 180, 1)

CMAP_MAGENTA = LinearSegmentedColormap.from_list('magenta', ['white', 'magenta'])

# ── EDX hyper-parameters  (arXiv:2606.10547) ──────────────────────────────
NUM_ITER_EDX      = 5000
LR_EDX            = 4e-4
NOISE_REG_EDX     = 0.04
LAMBDA_TV_EDX     = 1e-10
STD_INP_NOISE_EDX = 1. / 10.
LOSS_TYPE_EDX     = 'L2'
USE_AMP_EDX       = True
SAVE_EVERY_EDX    = 200
UD_FILTERS_EDX    = [16, 32, 64, 128, 256]
SKIP_FILTERS_EDX  = [ 4,  8, 16,  32,  64]

# ── EELS hyper-parameters  (arXiv:2606.10893) ─────────────────────────────
NUM_ITER_EELS      = 1000
LR_EELS            = 4e-4
NOISE_REG_EELS     = 0.01
LAMBDA_TV_EELS     = 1e-10
STD_INP_NOISE_EELS = 1. / 10.
LOSS_TYPE_EELS     = 'L1'
USE_AMP_EELS       = False
SAVE_EVERY_EELS    = 100
UD_FILTERS_EELS    = [16, 32, 64, 128]
SKIP_FILTERS_EELS  = [ 4,  8, 16,  32]

def build_net(nbr, ud_filters, skip_filters, device=DEVICE):
    """Build CNN3D for `nbr` output channels."""
    n = len(ud_filters)
    return CNN3D(
        nbr=nbr, input_shape=INPUT_DEPTH,
        down_filters=ud_filters, up_filters=ud_filters, skip_filters=skip_filters,
        down_kernels=(3,)*n, up_kernels=(3,)*n, skip_kernels=(1,)*n,
        up_mode='trilinear', pad_mode='reflect',
    ).to(device)

---
## A — EDX   |  Cube + sphere hole + ellipsoid hole  |  −40 : 5 : +40

### A.1  Build phantom
Volume shape `(Y, X, Z)` — **Y = tilt axis** (axis 0, depth),  **X** = beam direction at 0° tilt (axis 1),  **Z** = missing-wedge elongation direction (axis 2).

Cube + sphere hole + ellipsoid hole; sphere and ellipsoid features sit at **Y = 70**.

In [ ]:
NY_EDX = NX_EDX = NZ_EDX = 160

XC_EDX = NX_EDX // 2
ZC_EDX = NZ_EDX // 2 + 6
HALF   = 30
Y_FEAT = 70

data1_edx = np.zeros((NY_EDX, NX_EDX, NZ_EDX), dtype=np.float32)  # cube
data2_edx = np.zeros_like(data1_edx)                               # ellipsoid
data3_edx = np.zeros_like(data1_edx)                               # sphere

data1_edx[:, XC_EDX-HALF:XC_EDX+HALF, ZC_EDX-HALF:ZC_EDX+HALF] = 25.0

paint_sphere(data3_edx, Y_FEAT, XC_EDX-4,  ZC_EDX+15, r=9,  val=15.0)
paint_sphere(data1_edx, Y_FEAT, XC_EDX-4,  ZC_EDX+15, r=9,  val=0.0)

paint_ellipsoid(data2_edx, Y_FEAT, XC_EDX, ZC_EDX-14, ry=6, rx=19, rz=6, val=20.0)
paint_ellipsoid(data1_edx, Y_FEAT, XC_EDX, ZC_EDX-14, ry=6, rx=19, rz=6, val=0.0)

Visualization: 

In [ ]:
def rgb_slice_edx(sl, gamma=0.9):
    def _n01(x, hi): return np.clip(x / (hi + 1e-12), 0, 1).astype(np.float32)
    rgb = np.stack([
        _n01(data1_edx[sl], data1_edx.max()),
        _n01(data2_edx[sl], data2_edx.max()),
        _n01(data3_edx[sl], data3_edx.max()),
    ], axis=-1)
    return np.clip(rgb, 0, 1) ** gamma

SLICES_EDX = [
    (np.s_[Y_FEAT, :, :],  f'XZ  (y={Y_FEAT}  ⊥ tilt axis — MW elongation along Z)'),
    (np.s_[:, XC_EDX, :],  f'YZ  (x={XC_EDX}  |  Y=tilt axis, Z=MW direction)'),
    (np.s_[:, :, ZC_EDX],  f'YX  (z={ZC_EDX}  |  Y=tilt axis, no MW)'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=120)
for ax, (sl, lbl) in zip(axes, SLICES_EDX):
    ax.imshow(rgb_slice_edx(sl), interpolation="nearest")
    ax.set_title(lbl, fontsize=9); ax.axis('off')
plt.suptitle('EDX phantom\nred = cube  |  green = ellipsoid  |  blue = sphere')
plt.tight_layout(); plt.show()

### A.2  Reference reconstruction  (0 : 1 : 179, 180 proj.)

In [ ]:
data1_edx_t = torch.from_numpy(data1_edx).float().to(DEVICE)
data2_edx_t = torch.from_numpy(data2_edx).float().to(DEVICE)
data3_edx_t = torch.from_numpy(data3_edx).float().to(DEVICE)
CH_NAMES_EDX = ['cube', 'ellipsoid', 'sphere']

rad_ref_edx = Radon3D(depth=NY_EDX, size=NX_EDX, angle=np.deg2rad(THETA_REF), device=DEVICE)
print('SIRT reference EDX — 3 channels (180 proj.)...')
with torch.no_grad():
    sirt_ref_edx_1 = rad_ref_edx.backward_sirt_ts(rad_ref_edx(data1_edx_t), num_iters=100)
    sirt_ref_edx_2 = rad_ref_edx.backward_sirt_ts(rad_ref_edx(data2_edx_t), num_iters=100)
    sirt_ref_edx_3 = rad_ref_edx.backward_sirt_ts(rad_ref_edx(data3_edx_t), num_iters=100)

sirt_ref_edx_list = [sirt_ref_edx_1, sirt_ref_edx_2, sirt_ref_edx_3]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, sirt_ref, ch_name in zip(axes, sirt_ref_edx_list, CH_NAMES_EDX):
    ax.imshow(norm_slice(sirt_ref, np.s_[Y_FEAT, :, :]))
    ax.set_title(f'SIRT ref (180 proj.) — {ch_name}  [XZ y={Y_FEAT}]', fontsize=9)
    ax.axis('off')
plt.suptitle('EDX — SIRT reference (180 proj.) — XZ feature plane')
plt.tight_layout(); plt.show()

### A.3  Limited tilt series  (−40 : 5 : +40)  +  SIRT

In [ ]:
THETA_EDX = np.arange(90-40, 90+40+1, 5)
print(f'Physical: {THETA_EDX[0]-90:.0f} : 5 : {THETA_EDX[-1]-90:.0f}   ({len(THETA_EDX)} proj.)')

rad_edx = Radon3D(depth=NY_EDX, size=NX_EDX, angle=np.deg2rad(THETA_EDX), device=DEVICE)

print('Computing sinograms and SIRT for 3 channels...')
with torch.no_grad():
    sino_edx_1 = rad_edx(data1_edx_t)
    sino_edx_2 = rad_edx(data2_edx_t)
    sino_edx_3 = rad_edx(data3_edx_t)

    sirt_edx_1 = rad_edx.backward_sirt_ts(sino_edx_1, num_iters=100)
    sirt_edx_2 = rad_edx.backward_sirt_ts(sino_edx_2, num_iters=100)
    sirt_edx_3 = rad_edx.backward_sirt_ts(sino_edx_3, num_iters=100)

sirt_edx_list = [sirt_edx_1, sirt_edx_2, sirt_edx_3]

print('Computing full-range sinograms for MW visualisation...')
with torch.no_grad():
    sino_gt_edx_1 = rad_ref_edx(data1_edx_t)
    sino_gt_edx_2 = rad_ref_edx(data2_edx_t)
    sino_gt_edx_3 = rad_ref_edx(data3_edx_t)

sino_emb_edx = [embed_sino(s, THETA_EDX) for s in [sino_edx_1, sino_edx_2, sino_edx_3]]

# Sinogram comparison at Y_FEAT
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
for ch_idx, (ch_name, sino_gt, sino_emb) in enumerate(zip(
        CH_NAMES_EDX,
        [sino_gt_edx_1, sino_gt_edx_2, sino_gt_edx_3],
        sino_emb_edx)):
    gt_slice  = sino_gt[Y_FEAT].cpu().numpy()
    emb_slice = sino_emb[Y_FEAT]
    for col, (data, lbl) in enumerate([
        (gt_slice,  f'GT sinogram (180 proj.) — {ch_name}'),
        (emb_slice, f'MW sinogram (17/180 proj.) — {ch_name}  [±40°]'),
    ]):
        axes[ch_idx, col].imshow(data, aspect='auto', extent=[0, NX_EDX, 90, -90])
        axes[ch_idx, col].set(title=lbl, xlabel='detector X (px)', ylabel='tilt angle (°)')
plt.suptitle(f'EDX — GT vs Missing-Wedge sinograms at Y={Y_FEAT}')
plt.tight_layout(); plt.show()

# GT vs SIRT at feature XZ plane
sl_xz = np.s_[Y_FEAT, :, :]
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
gts_edx = [data1_edx, data2_edx, data3_edx]
for ch_idx, (ch_name, gt, sirt) in enumerate(zip(CH_NAMES_EDX, gts_edx, sirt_edx_list)):
    axes[0, ch_idx].imshow(norm_slice(gt,   sl_xz)); axes[0, ch_idx].set_title(f'GT — {ch_name}')
    axes[1, ch_idx].imshow(norm_slice(sirt, sl_xz)); axes[1, ch_idx].set_title(f'SIRT (17 proj.) — {ch_name}')
    axes[0, ch_idx].axis('off'); axes[1, ch_idx].axis('off')
plt.suptitle(f'EDX — XZ plane at Y={Y_FEAT}  (Z=MW direction — elongation visible horizontally)')
plt.tight_layout(); plt.show()

### A.4  DIPm-TV

In [ ]:
sino_edx_np = np.stack([
    s.cpu().numpy().transpose(1, 0, 2)
    for s in [sino_edx_1, sino_edx_2, sino_edx_3]
])  # → (3, 17, Y=160, 160)

sino_edx_t, x_min_edx, x_max_edx = preprocess_sinograms(sino_edx_np, THETA_EDX, device=DEVICE)
# → (3, Y=160, 17, 160)   x_min/x_max: (3,) per-channel scale factors

sirt_edx_stack = torch.stack([sirt_edx_1, sirt_edx_2, sirt_edx_3])   # (3, Y, X, Z)

net_edx = build_net(nbr=3, ud_filters=UD_FILTERS_EDX, skip_filters=SKIP_FILTERS_EDX)

print(f'DIPm-TV  EDX  [arXiv:2606.10547]  3-channel  ({NUM_ITER_EDX} iters)')
print(f'  LR={LR_EDX}  NOISE_REG={NOISE_REG_EDX}  LAMBDA_TV={LAMBDA_TV_EDX}')
print(f'  STD_INP_NOISE={STD_INP_NOISE_EDX}  loss={LOSS_TYPE_EDX}  USE_AMP={USE_AMP_EDX}')
print(f'  net depth={len(UD_FILTERS_EDX)}  filters={UD_FILTERS_EDX}')

iter_out_edx, loss_edx = run_dipm_tv(
    net_edx, rad_edx, sino_edx_t,
    sirt_vol      = sirt_edx_stack,
    num_iter      = NUM_ITER_EDX,
    input_depth   = INPUT_DEPTH,
    depth         = NY_EDX,
    img_size      = NX_EDX,
    lr            = LR_EDX,
    noise_reg     = NOISE_REG_EDX,
    lambda_tv     = LAMBDA_TV_EDX,
    loss_type     = LOSS_TYPE_EDX,
    std_inp_noise = STD_INP_NOISE_EDX,
    weight_decay  = 0.0,
    use_amp       = USE_AMP_EDX,
    plot_every    = 100,
    plot_slice    = Y_FEAT,
    plot_axis     = 0,
    save_every    = SAVE_EVERY_EDX,
    device        = DEVICE,
)

### A.5  Results  —  GT | SIRT ref | SIRT limited | DIPm-TV

In [ ]:
rec_edx_norm = iter_out_edx[-1][1]   # (3, Y, X, Z)

rec_edx = rec_edx_norm.copy()
for i in range(rec_edx.shape[0]):
    rec_edx[i] = rec_edx[i] * (x_max_edx[i] - x_min_edx[i]) + x_min_edx[i]

panel_results(
    vol_lists    = [
        [data1_edx, data2_edx, data3_edx],
        sirt_ref_edx_list,
        sirt_edx_list,
        [rec_edx[0], rec_edx[1], rec_edx[2]],
    ],
    method_names = ['GT', 'SIRT ref\n(180 proj.)', 'SIRT\n(17 proj.)', 'DIPm-TV'],
    gt_list      = [data1_edx, data2_edx, data3_edx],
    rows = [
        ('cube',
         lambda sv, p: rgb_channel(sv[0], 'red', p_hi=p),
         [0]),
        ('ellipsoid + sphere',
         lambda sv, p: rgb_two_ch(sv[1], sv[2], 'green', 'blue', p_hi=p),
         [1, 2]),
    ],
    axis=0, slice_idx=Y_FEAT, perc=(1, 99),
    title=f'EDX  [arXiv:2606.10547]  —  XZ slice at Y={Y_FEAT}',
)

plt.figure(figsize=(5, 3))
plt.semilogy(loss_edx)
plt.title('EDX training loss'); plt.xlabel('iteration'); plt.tight_layout(); plt.show()

---
## B — EELS  |  Core-shell nanocube with pore  |  −70 : 17.5 : +70

### B.1  Build phantom
Volume shape `(Y, X, Z)` — **Y = tilt axis** (axis 0),  **X** = beam at 0° tilt (axis 1),  **Z** = MW elongation direction (axis 2).

3-layer nanocube in 2 channels (arXiv:2606.10893):

| Channel | Region | Value |
|---|---|---|
| **data1** | Hollow main shell (`size_core` → `size_shell`) | `shell_value = 10` |
| **data2** | Thin outer shell (`size_shell` → `size_shell + thin_out`) + solid core (≤ `size_core`) | `core_value = 20` |

In [ ]:
NY_EELS = NX_EELS = NZ_EELS = 160
c = NY_EELS // 2
CY_EELS = CX_EELS = CZ_EELS = c

size_shell       = 40
size_core        = 20
thin_out         = 3
size_shell_outer = size_shell + thin_out
shell_value      = 10.0
core_value       = 20.0

ROTATIONS = [(5, (0,1)), (7, (1,2)), (13, (0,2))]

# DATA1: hollow main shell
cube_shell = np.zeros((NY_EELS, NX_EELS, NZ_EELS), dtype=bool)
cube_core  = np.zeros((NY_EELS, NX_EELS, NZ_EELS), dtype=bool)
cube_shell[c-size_shell:c+size_shell, c-size_shell:c+size_shell, c-size_shell:c+size_shell] = True
cube_core [c-size_core :c+size_core,  c-size_core :c+size_core,  c-size_core :c+size_core ] = True
main_shell_mask = cube_shell & (~cube_core)

data1_eels = np.zeros((NY_EELS, NX_EELS, NZ_EELS), dtype=np.float32)
data1_eels[main_shell_mask] = shell_value

# DATA2: thin outer shell + solid core
cube_outer = np.zeros((NY_EELS, NX_EELS, NZ_EELS), dtype=bool)
cube_outer[c-size_shell_outer:c+size_shell_outer,
           c-size_shell_outer:c+size_shell_outer,
           c-size_shell_outer:c+size_shell_outer] = True
thin_shell_mask = cube_outer & (~cube_shell)

data2_eels = np.zeros((NY_EELS, NX_EELS, NZ_EELS), dtype=np.float32)
data2_eels[thin_shell_mask] = core_value
data2_eels[c-size_core:c+size_core, c-size_core:c+size_core, c-size_core:c+size_core] = core_value

data1_eels = rotate_rebinarize(data1_eels, ROTATIONS, thr=shell_value*0.5, high=shell_value)
data2_eels = rotate_rebinarize(data2_eels, ROTATIONS, thr=core_value *0.5, high=core_value)

In [ ]:
SLICES_EELS = [
    (np.s_[CY_EELS, :, :], f'XZ  y={CY_EELS}  [⊥ tilt axis — MW elongation along Z]'),
    (np.s_[:, CX_EELS, :],  f'YZ  x={CX_EELS}  [Y=tilt axis | Z=MW direction]'),
    (np.s_[:, :, CZ_EELS],  f'YX  z={CZ_EELS}  [Y=tilt axis | no MW]'),
]

def rgb_slice_eels(sl, gamma=0.9):
    def _n01(x, hi): return np.clip(x / (hi + 1e-12), 0, 1).astype(np.float32)
    m = _n01(data1_eels[sl], data1_eels.max())
    g = _n01(data2_eels[sl], data2_eels.max())
    return np.clip(np.stack([m, g, m], axis=-1), 0, 1) ** gamma

fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=120)
for ax, (sl, lbl) in zip(axes, SLICES_EELS):
    ax.imshow(rgb_slice_eels(sl), interpolation="nearest")
    ax.set_title(lbl, fontsize=9); ax.axis('off')
plt.suptitle('EELS phantom\nmagenta = hollow shell  |  green = thin outer shell + core')
plt.tight_layout(); plt.show()

### B.2  Reference reconstruction  (0 : 1 : 179, 180 proj.)

In [ ]:
data1_eels_t = torch.from_numpy(data1_eels).float().to(DEVICE)
data2_eels_t = torch.from_numpy(data2_eels).float().to(DEVICE)
CH_NAMES_EELS = ['hollow shell', 'thin outer shell + core']

rad_ref_eels = Radon3D(depth=NY_EELS, size=NX_EELS, angle=np.deg2rad(THETA_REF), device=DEVICE)
print('SIRT reference EELS — 2 channels (180 proj.)...')
with torch.no_grad():
    sirt_ref_eels_1 = rad_ref_eels.backward_sirt_ts(rad_ref_eels(data1_eels_t), num_iters=100)
    sirt_ref_eels_2 = rad_ref_eels.backward_sirt_ts(rad_ref_eels(data2_eels_t), num_iters=100)

sirt_ref_eels_list = [sirt_ref_eels_1, sirt_ref_eels_2]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, sirt_ref, ch_name in zip(axes, sirt_ref_eels_list, CH_NAMES_EELS):
    ax.imshow(norm_slice(sirt_ref, np.s_[CY_EELS, :, :]))
    ax.set_title(f'SIRT ref (180 proj.) — {ch_name}  [XZ y={CY_EELS}]', fontsize=9)
    ax.axis('off')
plt.suptitle('EELS — SIRT reference (180 proj.) — XZ mid plane')
plt.tight_layout(); plt.show()

### B.3  Limited tilt series  (−70 : 17.5 : +70)  +  SIRT

In [ ]:
THETA_EELS = np.arange(90-70, 90+70+1, 17.5)
print(f'EELS  physical: {THETA_EELS[0]-90:.1f} : 17.5 : {THETA_EELS[-1]-90:.1f}   ({len(THETA_EELS)} proj.)')

rad_eels = Radon3D(depth=NY_EELS, size=NX_EELS, angle=np.deg2rad(THETA_EELS), device=DEVICE)

print('Computing sinograms and SIRT for 2 channels...')
with torch.no_grad():
    sino_eels_1 = rad_eels(data1_eels_t)
    sino_eels_2 = rad_eels(data2_eels_t)
    sirt_eels_1 = rad_eels.backward_sirt_ts(sino_eels_1, num_iters=100)
    sirt_eels_2 = rad_eels.backward_sirt_ts(sino_eels_2, num_iters=100)

sirt_eels_list = [sirt_eels_1, sirt_eels_2]

print('Computing full-range sinograms for MW visualisation...')
with torch.no_grad():
    sino_gt_eels_1 = rad_ref_eels(data1_eels_t)
    sino_gt_eels_2 = rad_ref_eels(data2_eels_t)

sino_emb_eels = [embed_sino(s, THETA_EELS) for s in [sino_eels_1, sino_eels_2]]

# Sinogram comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ch_idx, (ch_name, sino_gt, sino_emb) in enumerate(zip(
        CH_NAMES_EELS, [sino_gt_eels_1, sino_gt_eels_2], sino_emb_eels)):
    gt_slice  = sino_gt[CY_EELS].cpu().numpy()
    emb_slice = sino_emb[CY_EELS]
    for col, (data, lbl) in enumerate([
        (gt_slice,  f'GT sinogram (180 proj.) — {ch_name}'),
        (emb_slice, f'MW sinogram (9/180 proj.) — {ch_name}  [±70°]'),
    ]):
        axes[ch_idx, col].imshow(data, aspect='auto', extent=[0, NX_EELS, 90, -90])
        axes[ch_idx, col].set(title=lbl, xlabel='detector X (px)', ylabel='tilt angle (°)')
plt.suptitle(f'EELS — GT vs Missing-Wedge sinograms at Y={CY_EELS}')
plt.tight_layout(); plt.show()

# GT vs SIRT
sl_xz = np.s_[CY_EELS, :, :]
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
gts_eels = [data1_eels, data2_eels]
for ch_idx, (ch_name, gt, sirt) in enumerate(zip(CH_NAMES_EELS, gts_eels, sirt_eels_list)):
    axes[0, ch_idx].imshow(norm_slice(gt,   sl_xz)); axes[0, ch_idx].set_title(f'GT — {ch_name}')
    axes[1, ch_idx].imshow(norm_slice(sirt, sl_xz)); axes[1, ch_idx].set_title(f'SIRT (9 proj.) — {ch_name}')
    axes[0, ch_idx].axis('off'); axes[1, ch_idx].axis('off')
plt.suptitle(f'EELS — XZ plane at Y={CY_EELS}  (Z=MW direction)')
plt.tight_layout(); plt.show()

### B.4  DIPm-TV

In [ ]:
sino_eels_np = np.stack([
    s.cpu().numpy().transpose(1, 0, 2)
    for s in [sino_eels_1, sino_eels_2]
])

sino_eels_t, x_min_eels, x_max_eels = preprocess_sinograms(sino_eels_np, THETA_EELS, device=DEVICE)

sirt_eels_stack = torch.stack([sirt_eels_1, sirt_eels_2])

net_eels = build_net(nbr=2, ud_filters=UD_FILTERS_EELS, skip_filters=SKIP_FILTERS_EELS)

print(f'DIPm-TV  EELS  [arXiv:2606.10893]  2-channel  ({NUM_ITER_EELS} iters)')
print(f'  LR={LR_EELS}  NOISE_REG={NOISE_REG_EELS}  LAMBDA_TV={LAMBDA_TV_EELS}')
print(f'  STD_INP_NOISE={STD_INP_NOISE_EELS}  loss={LOSS_TYPE_EELS}  USE_AMP={USE_AMP_EELS}')
print(f'  net depth={len(UD_FILTERS_EELS)}  filters={UD_FILTERS_EELS}')

iter_out_eels, loss_eels = run_dipm_tv(
    net_eels, rad_eels, sino_eels_t,
    sirt_vol      = sirt_eels_stack,
    num_iter      = NUM_ITER_EELS,
    input_depth   = INPUT_DEPTH,
    depth         = NY_EELS,
    img_size      = NX_EELS,
    lr            = LR_EELS,
    noise_reg     = NOISE_REG_EELS,
    lambda_tv     = LAMBDA_TV_EELS,
    loss_type     = LOSS_TYPE_EELS,
    std_inp_noise = STD_INP_NOISE_EELS,
    weight_decay  = 0.0,
    use_amp       = USE_AMP_EELS,
    plot_every    = 10,
    plot_slice    = CY_EELS,
    plot_axis     = 0,
    save_every    = SAVE_EVERY_EELS,
    device        = DEVICE,
)

### B.5  Results SIMU MW  —  GT | SIRT ref | SIRT limited | DIPm-TV

In [ ]:
rec_eels_norm = iter_out_eels[-1][1]   # (2, Y, X, Z)

rec_eels = rec_eels_norm.copy()
for i in range(rec_eels.shape[0]):
    rec_eels[i] = rec_eels[i] * (x_max_eels[i] - x_min_eels[i]) + x_min_eels[i]

panel_results(
    vol_lists    = [
        [data1_eels, data2_eels],
        sirt_ref_eels_list,
        sirt_eels_list,
        [rec_eels[0], rec_eels[1]],
    ],
    method_names = ['GT', 'SIRT ref\n(180 proj.)', 'SIRT\n(9 proj.)', 'DIPm-TV'],
    gt_list      = [data1_eels, data2_eels],
    rows = [
        ('shell',
         lambda sv, p: rgb_channel(sv[0], 'magenta', p_hi=p),
         [0]),
        ('core',
         lambda sv, p: rgb_channel(sv[1], 'green', p_hi=p),
         [1]),
    ],
    axis=0, slice_idx=CY_EELS, perc=(1, 99),
    title=f'EELS  [arXiv:2606.10893]  —  XZ slice at Y={CY_EELS}',
)

plt.figure(figsize=(5, 3))
plt.semilogy(loss_eels)
plt.title('EELS training loss'); plt.xlabel('iteration'); plt.tight_layout(); plt.show()